# MCSR Ranked Playoff Prediction
**DADS 7275 — Final Project**

## Overview
This notebook builds a machine learning system to predict outcomes in the **MCSR Ranked** (Minecraft Speedrunning Ranked) playoff tournament. MCSR Ranked is a competitive Minecraft speedrunning platform where players are rated by an Elo system and compete in seasonal 16-player single-elimination brackets.

### Research Questions
1. Can player match-history features predict head-to-head playoff matchup winners?
2. Which features are most predictive of playoff success?
3. What player archetypes exist in the competitive player pool?
4. Who are the projected Season 10 playoff contenders?

### Dataset
- **9 seasons** of ranked match history (Seasons 1–9)
- **16 players** per playoff bracket, single-elimination
- Features: Elo rating, win rate, recent form, completion speed, tournament pedigree, head-to-head history, LCQ qualifier status, and tournament performance delta

### Models
| Model | Task | Evaluation |
|-------|------|------------|
| Logistic Regression | Binary: p1 beats p2? | S9 hold-out accuracy |
| LDA | Multi-class: playoff outcome tier | S9 hold-out accuracy |
| K-Means | Unsupervised: player archetypes | Silhouette score |
| PCA / t-SNE | Dimensionality reduction / visualization | Explained variance |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, silhouette_score
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

In [ ]:
# ── Paths & Constants ────────────────────────────────────────────────────────
BASE_DIR = os.path.abspath('.')          # run from project root
RAW      = os.path.join(BASE_DIR, 'data', 'raw')
OUT_DIR  = os.path.join(BASE_DIR, 'data', 'processed')
os.makedirs(OUT_DIR, exist_ok=True)

SEASONS = list(range(1, 10))

PLAYOFF_PLAYERS = {
    1: ['silverrruns','dandannyboy','Oxidiot','Reignex','priffie','orachi_','lowk3y_','7rowl','doogile','Ancoboyy','pulsar32','Ranik_','MoleyG','CroProYT','AutomattPL','Dylqn'],
    2: ['lowk3y_','CroProYT','dandannyboy','doogile','7rowl','kW1st','priffie','dwoh','silverrruns','Emillk','bing_pigs','drx6','Ancoboyy','Ranik_','Oxidiot','AutomattPL'],
    3: ['7rowl','Ancoboyy','dandannyboy','doogile','hackingnoises','lowk3y_','Oxidiot','priffie','ANJOUU','AutomattPL','BeefSalad','Bloonskiller','loodlow','paplerr','v_strid','autoqualler'],
    4: ['7rowl','Ancoboyy','dandannyboy','doogile','AutomattPLUS','hackingnoises','Hinart','lowk3y_','Oxidot','paplerr','priffie','silverrruns','ANJOUU','bing_pigs','Cube1337x','v_strid'],
    5: ['7rowl','Ancoboyy','BeefSalad','bing_pigs','doogile','AutomattPLUS','hackingnoises','lowk3y_','Oxidiot','silverrruns','TUDORULE','v_strid','Aquacorde','dandannyboy','KenanKardes','pulsar32'],
    6: ['7rowl','Ayreliaa','BeefSalad','bing_pigs','doogile','AutomattPLUS','Feinberg','hackingnoises','lowk3y_','MrBudgiee','Oxidiot','silverrruns','dandannyboy','Erikfzf','ogurikappa','TUDORULE'],
    7: ['7rowl','Ancoboyy','Aquacorde','BadGamer','BeefSalad','bing_pigs','doogile','Feinberg','Infume','lowk3y_','priffie','retropog','r7sD4fH6jK0wY5uB','hackingnoises','Oxidiot','silverrruns'],
    8: ['7rowl','Aquacorde','BeefSalad','bing_pigs','DARVY__X1','doogile','edcr','Feinberg','Infume','lowk3y_','Ranik_','silverrruns','hackingnoises','KenanKardes','TUDORULE','v_strid'],
    9: ['Feinberg','Infume','edcr','steez','hackingnoises','Aquacorde','nhb_','silverrruns','Pinne','BeefSalad','nahhann','lowk3y_','doogile','HDMICables','bing_pigs','BlazeMind'],
}

S10_POOL = {
    'Infume','edcr','doogile','Feinberg','7rowl','bing_pigs',
    'nahhann','BlazeMind','Aquacorde','silverrruns','BeefSalad',
    'meebie','hackingnoises','steez','nhb_','Ancoboyy',
}

TIER_ORDER = ['champion','finalist','top4','qf_exit','r1_exit']
SEASON_WEIGHTS = {1:1, 2:1, 3:1, 4:2, 5:2, 6:3, 7:4, 8:6}

FEATURE_NAMES = [
    'Elo diff','WinRate diff','RecentWR diff','Consistency diff',
    'AvgTime diff (s)','DeepRun diff','Champion diff','Finalist diff',
    'BestTime diff (s)','ForfeitRate diff','EloMomentum diff',
    'LCQ Flag diff','Tournament Delta diff','H2H WinRate',
]
PLAYER_FEAT_COLS = [
    'elo','win_rate','recent_wr','consistency','avg_time_ms',
    'deep_run_score','champion_count','finalist_count',
    'best_time_ms','forfeit_rate','elo_momentum',
    'lcq_flag','avg_tournament_delta',
]

print(f'RAW data directory : {RAW}')
print(f'Output directory   : {OUT_DIR}')

---
## 1. Data Loading
Raw data is stored as JSON files scraped from the MCSR Ranked API:
- `season_X_matches.json` — all ranked match records for that season
- `season_X_playoffs.json` — bracket seeding and final results
- `season_X_h2h.json` — aggregated head-to-head records
- `all_playoff_results.json` — consolidated multi-season results

In [ ]:
def load_all_matches() -> pd.DataFrame:
    """Load all ranked match rows (seasons 1–9) into a flat DataFrame."""
    frames = []
    for s in SEASONS:
        fpath = os.path.join(RAW, f'season_{s}_matches.json')
        if not os.path.exists(fpath): continue
        with open(fpath) as f:
            matches = json.load(f)
        for m in matches:
            players = m.get('players', [])
            result  = m.get('result') or {}
            winner_uuid = result.get('uuid')
            win_time    = result.get('time')
            forfeited   = m.get('forfeited', False)
            if len(players) < 2: continue
            p1, p2 = players[0], players[1]
            frames.append({
                'match_id': m.get('id'),
                'season':   m.get('season', s),
                'date':     m.get('date'),
                'p1_nick':  p1.get('nickname'), 'p1_uuid': p1.get('uuid'), 'p1_elo': p1.get('eloRate'),
                'p2_nick':  p2.get('nickname'), 'p2_uuid': p2.get('uuid'), 'p2_elo': p2.get('eloRate'),
                'winner_uuid': winner_uuid, 'win_time_ms': win_time, 'forfeited': forfeited,
            })
    df = pd.DataFrame(frames).drop_duplicates('match_id')
    df['p1_won'] = df['winner_uuid'] == df['p1_uuid']
    return df


def load_playoff_results() -> dict:
    path = os.path.join(RAW, 'all_playoff_results.json')
    with open(path) as f:
        results = {int(k): v for k, v in json.load(f).items()}
    # Confirmed S9 ground truth
    results[9] = {
        'champion': 'hackingnoises', 'finalist': 'doogile',
        'top4':    ['Pinne','Infume'],
        'qf_exit': ['steez','Aquacorde','lowk3y_','BlazeMind'],
        'r1_exit': ['edcr','Feinberg','nhb_','silverrruns','BeefSalad','nahhann','HDMICables','bing_pigs'],
    }
    return results


def load_h2h_csv() -> pd.DataFrame:
    """Load aggregated H2H records from all_player_stats.csv + season_X_h2h.json."""
    stats_path = os.path.join(RAW, 'all_player_stats.csv')
    if not os.path.exists(stats_path):
        return pd.DataFrame()
    stats = pd.read_csv(stats_path)
    uuid_to_nick = dict(zip(stats['uuid'], stats['nickname']))
    rows = []
    for s in SEASONS:
        path = os.path.join(RAW, f'season_{s}_h2h.json')
        if not os.path.exists(path): continue
        with open(path) as f:
            records = json.load(f)
        for r in records:
            p1, p2 = r.get('player1'), r.get('player2')
            ranked = r.get('results_ranked') or {}
            total  = ranked.get('total', 0)
            if total == 0: continue
            uuid_wins = {k: v for k, v in ranked.items() if k != 'total'}
            p1_uuid = next((k for k, n in uuid_to_nick.items()
                            if isinstance(n, str) and n.lower() == (p1 or '').lower()), None)
            p1_wins = uuid_wins.get(p1_uuid, 0) if p1_uuid else 0
            rows.append({'season': s, 'player1': p1, 'player2': p2,
                         'total': total, 'p1_wins': p1_wins,
                         'p2_wins': total - p1_wins,
                         'p1_winrate': p1_wins / total})
    return pd.DataFrame(rows)


# ── Load everything ──────────────────────────────────────────────────────────
df_matches      = load_all_matches()
playoff_results = load_playoff_results()
h2h_df          = load_h2h_csv()

print(f'Matches loaded       : {len(df_matches):,} unique matches')
print(f'Seasons covered      : {sorted(df_matches["season"].unique())}')
print(f'Playoff seasons      : {sorted(playoff_results.keys())}')
print(f'H2H pairs loaded     : {len(h2h_df)}')
df_matches.head(3)

---
## 2. Data Cleaning & Quality
We check for missing values and confirm data integrity before building features.

In [ ]:
# ── Missing-value audit ───────────────────────────────────────────────────────
print('=== Match DataFrame — Missing Values ===')
mv = df_matches.isnull().sum()
print(mv[mv > 0].to_string())

print(f'\nTotal rows      : {len(df_matches):,}')
print(f'Forfeited games : {df_matches["forfeited"].sum():,}  ({df_matches["forfeited"].mean():.1%})')
print(f'Seasons in data : {df_matches["season"].nunique()}')
print(f'Unique players  : {pd.concat([df_matches["p1_nick"], df_matches["p2_nick"]]).nunique()}')

print('\n=== Matches per Season ===')
print(df_matches.groupby('season').size().rename('match_count').to_string())

In [ ]:
def build_player_season_features(df: pd.DataFrame) -> pd.DataFrame:
    """Per-player per-season stats restricted to playoff participants."""
    rows = []
    for season, players in PLAYOFF_PLAYERS.items():
        sdf = df[df['season'] == season]
        for player in players:
            as_p1 = sdf[sdf['p1_nick'].str.lower() == player.lower()]
            as_p2 = sdf[sdf['p2_nick'].str.lower() == player.lower()]
            wins  = int(as_p1['p1_won'].sum() + (~as_p2['p1_won']).sum())
            total = len(as_p1) + len(as_p2)
            if total == 0: continue
            times_p1 = as_p1[as_p1['p1_won'] & ~as_p1['forfeited']]['win_time_ms']
            times_p2 = as_p2[~as_p2['p1_won'] & ~as_p2['forfeited']]['win_time_ms']
            all_times = pd.concat([times_p1, times_p2]).dropna()

            # Elo at season end (safe — guard against all-NaN elo column)
            elo_series = pd.concat([
                as_p1[['date','p1_elo']].rename(columns={'p1_elo':'elo'}),
                as_p2[['date','p2_elo']].rename(columns={'p2_elo':'elo'})
            ]).sort_values('date')['elo'].dropna()
            elo_end = float(elo_series.iloc[-1]) if len(elo_series) > 0 else np.nan

            rows.append({
                'nickname':      player,
                'season':        season,
                'matches_played': total,
                'wins':          wins,
                'losses':        total - wins,
                'winrate':       wins / total,
                'avg_finish_ms': float(all_times.mean()) if len(all_times) else np.nan,
                'best_finish_ms': float(all_times.min()) if len(all_times) else np.nan,
                'consistency_score': (1 - all_times.std() / all_times.mean())
                                     if len(all_times) > 1 else np.nan,
                'forfeit_rate':  float((as_p1['forfeited'].sum() + as_p2['forfeited'].sum()) / total),
                'elo_at_season_end': elo_end,
            })
    return pd.DataFrame(rows)


def build_career_features(features: pd.DataFrame) -> pd.DataFrame:
    return features.groupby('nickname').agg(
        seasons_played=('season','nunique'),
        avg_winrate=('winrate','mean'),
        avg_finish_ms=('avg_finish_ms','mean'),
        best_finish_ms=('best_finish_ms','min'),
        avg_consistency=('consistency_score','mean'),
        avg_elo=('elo_at_season_end','mean'),
        avg_matches=('matches_played','mean'),
    ).reset_index()


player_features = build_player_season_features(df_matches)
career_features = build_career_features(player_features)

print(f'Player-season records : {len(player_features)}')
print(f'Unique players        : {player_features["nickname"].nunique()}')
print(f'\nFeature sample:')
player_features.head()


---
## 3. Exploratory Data Analysis
We visualize win rates, Elo ratings, completion speed, playoff appearances, and head-to-head matchup history to understand the distribution of player skill and the relationship between observable metrics.

In [ ]:
# ── Win Rate & Elo distributions ──────────────────────────────────────────────
def _s10_colors(names, c1='#e74c3c', c2='#3498db'):
    return [c1 if n in S10_POOL else c2 for n in names]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Win Rate — top 20
avg_wr = (player_features.groupby('nickname')['winrate']
          .mean().sort_values(ascending=False).head(20))
axes[0].bar(avg_wr.index, avg_wr.values * 100,
            color=_s10_colors(avg_wr.index), edgecolor='white')
axes[0].set_title('Average Win Rate — Top 20 Playoff Players')
axes[0].set_ylabel('Win Rate (%)')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(handles=[
    mpatches.Patch(color='#e74c3c', label='S10 Pool'),
    mpatches.Patch(color='#3498db', label='Historical only')
], loc='upper right')

# Playoff appearances
apps = (player_features.groupby('nickname')['season']
        .nunique().sort_values(ascending=False).head(20))
axes[1].bar(apps.index, apps.values,
            color=_s10_colors(apps.index), edgecolor='white')
axes[1].set_title('Playoff Appearances — Top 20 Players')
axes[1].set_ylabel('Number of Seasons')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(handles=[
    mpatches.Patch(color='#e74c3c', label='S10 Pool'),
    mpatches.Patch(color='#3498db', label='Historical only')
], loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_winrate_appearances.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Elo vs Win Rate scatter ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    player_features['elo_at_season_end'],
    player_features['winrate'],
    c=player_features['season'], cmap='plasma',
    s=player_features['matches_played'] * 2, alpha=0.6
)
plt.colorbar(sc, ax=ax, label='Season')
for size_val, label in [(10,'5 matches'),(40,'20 matches'),(80,'40 matches')]:
    ax.scatter([], [], s=size_val, c='gray', alpha=0.5, label=label)
ax.legend(title='Matches Played', loc='lower right')

valid = player_features.dropna(subset=['elo_at_season_end','winrate'])
corr  = valid['elo_at_season_end'].corr(valid['winrate'])
ax.set_title(f'Elo vs Win Rate  (Pearson r = {corr:.3f})\nColor = season, Size = matches played')
ax.set_xlabel('Elo at Season End')
ax.set_ylabel('Win Rate')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_elo_vs_winrate.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Pearson r (Elo ~ Win Rate): {corr:.4f}')

In [ ]:
# ── Finish time distribution & correlation heatmap ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Finish times by season
clean = player_features.dropna(subset=['avg_finish_ms']).copy()
clean['avg_finish_s'] = clean['avg_finish_ms'] / 1000
sns.boxplot(data=clean, x='season', y='avg_finish_s', ax=axes[0], palette='Set2')
axes[0].set_title('Average Finish Time Distribution per Season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Avg Finish Time (s)')
axes[0].text(0.01, 0.97, 'Box = playoff players | whiskers = 1.5x IQR',
             transform=axes[0].transAxes, fontsize=8, va='top', color='gray')

# Correlation heatmap
corr_cols = ['winrate','avg_finish_ms','consistency_score','forfeit_rate',
             'elo_at_season_end','matches_played']
corr_mat = player_features[corr_cols].corr()
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], linewidths=0.5)
axes[1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_timing_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Season 9 H2H Heatmap ─────────────────────────────────────────────────────
def plot_h2h_heatmap(h2h: pd.DataFrame, season: int = 9):
    sdf = h2h[h2h['season'] == season]
    if sdf.empty:
        print(f'No H2H data for season {season}')
        return
    players = sorted(set(sdf['player1'].tolist() + sdf['player2'].tolist()))
    mat = pd.DataFrame(np.nan, index=players, columns=players)
    for _, row in sdf.iterrows():
        mat.loc[row['player1'], row['player2']] = row['p1_winrate']
        mat.loc[row['player2'], row['player1']] = 1 - row['p1_winrate']
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(mat, annot=True, fmt='.0%', cmap='RdYlGn',
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title(f'Season {season} — Head-to-Head Win Rates\n'
                 '(green = row player wins more, red = row player loses more)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'h2h_heatmap_s{season}.png'), dpi=150, bbox_inches='tight')
    plt.show()

if not h2h_df.empty:
    plot_h2h_heatmap(h2h_df, season=9)
else:
    print('H2H data not available — skipping heatmap')

---
## 4. Player Clustering — K-Means Archetypes
We apply K-Means clustering to career-aggregated player features to discover natural player archetypes. The optimal number of clusters is selected using the **silhouette score**.

In [ ]:
CLUSTER_FEATURE_COLS = ['avg_winrate','avg_finish_ms','avg_consistency',
                        'avg_elo','seasons_played','avg_matches']

def _prep_cluster_matrix(career: pd.DataFrame):
    X_raw = career[CLUSTER_FEATURE_COLS].values.astype(float)
    imp   = SimpleImputer(strategy='median')
    scl   = StandardScaler()
    return scl.fit_transform(imp.fit_transform(X_raw)), imp, scl


# Silhouette score for k = 2..6
X_cluster, _imp, _scl = _prep_cluster_matrix(career_features)
sil_scores = {}
for k in range(2, 7):
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_cluster)
    sil_scores[k] = silhouette_score(X_cluster, lbl)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(sil_scores.keys()), list(sil_scores.values()), 'o-', color='#3498db', linewidth=2)
best_k = max(sil_scores, key=sil_scores.get)
ax.axvline(best_k, color='#e74c3c', linestyle='--', label=f'Best k={best_k}')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('K-Means: Silhouette Score by k')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'kmeans_silhouette.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Silhouette scores:')
for k, s in sil_scores.items():
    print(f'  k={k}: {s:.4f}  {"<-- best" if k == best_k else ""}')

In [ ]:
# ── Fit K-Means with best k and assign archetype labels ──────────────────────
def fit_kmeans(career: pd.DataFrame, k: int):
    X, imp, scl = _prep_cluster_matrix(career)
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    career = career.copy()
    career['cluster'] = labels

    # Sort clusters: label 0 = most elite (high Elo + win rate + fast finish)
    g = career.groupby('cluster').agg(
        avg_elo=('avg_elo','mean'), avg_winrate=('avg_winrate','mean'),
        avg_finish=('avg_finish_ms','mean'))
    def _norm(s): rng = s.max()-s.min(); return (s-s.min())/rng if rng > 0 else s*0
    prestige = (_norm(g['avg_elo'])*0.5 + _norm(g['avg_winrate'])*0.3
                + _norm(-g['avg_finish'])*0.2).sort_values(ascending=False)
    rank_map = {old: new for new, old in enumerate(prestige.index)}
    career['cluster'] = career['cluster'].map(rank_map)

    pca     = PCA(n_components=2, random_state=42)
    coords  = pca.fit_transform(X)
    career['pc1'] = coords[:, 0]
    career['pc2'] = coords[:, 1]
    career.attrs['pca_var'] = pca.explained_variance_ratio_
    sil = silhouette_score(X, labels)
    return career, km, sil


def label_archetypes(k: int) -> dict:
    names = ['Elite / Champion-tier','Consistent Veteran',
             'Rising Contender','Early Exit / Fringe','Occasional Qualifier']
    return {i: names[i] if i < len(names) else f'Cluster {i+1}' for i in range(k)}


career_clustered, km_model, sil_score = fit_kmeans(career_features, best_k)
archetype_map = label_archetypes(best_k)
colors_k = [plt.cm.tab10(i) for i in range(best_k)]

print(f'K-Means fitted  k={best_k}  silhouette={sil_score:.4f}')
for cl in sorted(career_clustered['cluster'].unique()):
    m = career_clustered[career_clustered['cluster'] == cl]
    print(f'  [{cl}] {archetype_map[cl]:30s}  n={len(m):3d}  '
          f'avg_elo={m["avg_elo"].mean():.0f}  win%={m["avg_winrate"].mean():.1%}')

---
## 5. Dimensionality Reduction — PCA & t-SNE
We use PCA and t-SNE to visualize the player feature space. **PCA** preserves global distance structure; **t-SNE** emphasizes local neighborhood similarity. Both are colored by K-Means cluster to confirm archetype coherence.

In [ ]:
# ── Compute t-SNE coords ──────────────────────────────────────────────────────
def compute_tsne(career: pd.DataFrame) -> pd.DataFrame:
    X, _, _ = _prep_cluster_matrix(career)
    perplexity = min(30, len(career) - 1)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, max_iter=1000)
    coords = tsne.fit_transform(X)
    career = career.copy()
    career['t1'] = coords[:, 0]
    career['t2'] = coords[:, 1]
    return career

career_clustered = compute_tsne(career_clustered)


def _annotate_selective(ax, career, x_col, y_col, priority_set):
    for _, row in career.iterrows():
        nick = row['nickname']
        x, y = row[x_col], row[y_col]
        if nick in priority_set:
            ax.annotate(nick, (x, y), xytext=(5,3), textcoords='offset points',
                        fontsize=8, fontweight='bold', color='black',
                        bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.6, lw=0))
        else:
            ax.annotate(nick, (x, y), xytext=(4,2), textcoords='offset points',
                        fontsize=6, color='#888888', alpha=0.75)


# ── Linked PCA + t-SNE plot ───────────────────────────────────────────────────
var      = career_clustered.attrs.get('pca_var', [0, 0])
priority = S10_POOL
s10_mask = career_clustered['nickname'].isin(S10_POOL)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle(f'Player Space: PCA (left) vs t-SNE (right) — K-Means k={best_k}\n'
             'Same color = same cluster. PCA preserves global distances; t-SNE preserves local neighborhoods.',
             fontsize=11)

for ax, (xc, yc), xlabel, ylabel in [
    (axes[0], ('pc1','pc2'), f'PC1 ({var[0]:.1%} var)', f'PC2 ({var[1]:.1%} var)'),
    (axes[1], ('t1','t2'),   't-SNE Dim 1', 't-SNE Dim 2'),
]:
    for cl in sorted(career_clustered['cluster'].unique()):
        mask = career_clustered['cluster'] == cl
        ax.scatter(career_clustered.loc[mask, xc], career_clustered.loc[mask, yc],
                   c=colors_k[cl], s=50, alpha=0.78,
                   label=f'C{cl}: {archetype_map[cl]}', edgecolors='white', lw=0.3)
    _annotate_selective(ax, career_clustered, xc, yc, priority)
    ax.scatter(career_clustered.loc[s10_mask, xc], career_clustered.loc[s10_mask, yc],
               marker='*', s=200, color='none', edgecolors='#c0392b', linewidths=1.3,
               label='S10 Pool', zorder=6)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'linked_pca_tsne.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'PCA explained variance: PC1={var[0]:.1%}, PC2={var[1]:.1%}, total={sum(var):.1%}')

---
## 6. Feature Engineering for Prediction
The prediction models require richer per-player feature snapshots computed **without data leakage**. Features for predicting Season X outcomes are computed from match data up to Season X−1 only.

We add three features beyond standard match statistics:
- **LCQ Flag** — did this player qualify via Last-Chance Qualifier (seed ≥ 12)? Public seeding info, not leakage.
- **Tournament Performance Delta** — historical average of `seed_rank − actual_place`. Positive = outperforms seeding.
- **H2H Win Rate** — direct head-to-head win rate vs. opponent, centered at 0.

In [ ]:
# ── Feature helper functions ──────────────────────────────────────────────────
def build_h2h_lookup(df: pd.DataFrame, season_filter: int) -> dict:
    sub = df[df['season'] <= season_filter][['p1_nick','p2_nick','p1_won']].dropna()
    wins, total = {}, {}
    for row in sub.itertuples(index=False):
        p1, p2, p1_won = row.p1_nick, row.p2_nick, row.p1_won
        total[(p1,p2)] = total.get((p1,p2), 0) + 1
        wins[ (p1,p2)] = wins.get( (p1,p2), 0) + int(p1_won)
        total[(p2,p1)] = total.get((p2,p1), 0) + 1
        wins[ (p2,p1)] = wins.get( (p2,p1), 0) + int(not p1_won)
    return {k: wins[k]/total[k] for k in total}


def load_lcq_by_season() -> dict:
    lcq = {}
    for s in SEASONS:
        path = os.path.join(RAW, f'season_{s}_playoffs.json')
        if not os.path.exists(path): lcq[s] = set(); continue
        try:
            with open(path) as f:
                data = json.load(f)['data']['data']
            lcq[s] = {p['nickname'] for p in data.get('players', [])
                      if p.get('seedNumber', 0) >= 12}
        except (KeyError, TypeError, json.JSONDecodeError):
            lcq[s] = set()
    return lcq


def load_delta_by_season() -> dict:
    deltas = {}
    for s in SEASONS:
        path = os.path.join(RAW, f'season_{s}_playoffs.json')
        if not os.path.exists(path): continue
        try:
            with open(path) as f:
                data = json.load(f)['data']['data']
            idx_to_nick = {p['seedNumber']: p['nickname'] for p in data.get('players', [])}
            season_d = {}
            for r in data.get('results', []):
                idx, place = r.get('player'), r.get('place')
                nick = idx_to_nick.get(idx)
                if nick and place is not None:
                    season_d[nick] = (idx + 1) - place   # seed_rank - actual_place
            deltas[s] = season_d
        except (KeyError, TypeError, json.JSONDecodeError):
            continue
    return deltas


lcq_by_season   = load_lcq_by_season()
delta_by_season = load_delta_by_season()
print('LCQ players per season:', {s: len(v) for s, v in lcq_by_season.items()})
print('Tournament deltas loaded for seasons:', sorted(delta_by_season.keys()))

In [ ]:
# ── Full per-player feature builder (14 features, leakage-safe) ───────────────
def _expand_tier(val):
    if val is None: return []
    return [val] if isinstance(val, str) else list(val)


def build_player_features_pred(
    df, players, playoff_results, season_filter,
    pedigree_cutoff=None, lcq_by_season=None,
    delta_by_season=None, lcq_season=None
):
    sub = df[df['season'] <= season_filter]
    if pedigree_cutoff is None:
        pedigree_cutoff = season_filter - 1
    ped = {k: v for k, v in playoff_results.items() if k <= pedigree_cutoff}
    records = []
    for nick in players:
        as_p1 = sub[sub['p1_nick'] == nick]
        as_p2 = sub[sub['p2_nick'] == nick]
        wins   = int(as_p1['p1_won'].sum() + (~as_p2['p1_won']).sum())
        losses = int((~as_p1['p1_won']).sum() + as_p2['p1_won'].sum())
        total  = wins + losses
        win_rate = wins / total if total > 0 else np.nan

        times_p1 = as_p1[as_p1['p1_won'] & ~as_p1['forfeited']]['win_time_ms']
        times_p2 = as_p2[~as_p2['p1_won'] & ~as_p2['forfeited']]['win_time_ms']
        all_times = pd.concat([times_p1, times_p2]).dropna()
        avg_time  = float(all_times.mean()) if len(all_times) > 0 else np.nan
        best_time = float(all_times.min())  if len(all_times) > 0 else np.nan
        std_time  = float(all_times.std())  if len(all_times) > 1 else np.nan
        consistency = 1.0 / (std_time/1000 + 1) if std_time and not np.isnan(std_time) else np.nan

        all_m = pd.concat([
            as_p1[['date','p1_won']].rename(columns={'p1_won':'won'}),
            as_p2[['date','p1_won']].rename(columns={'p1_won':'won'}).assign(won=lambda x: ~x['won'])
        ]).sort_values('date').tail(20)
        recent_wr = float(all_m['won'].mean()) if len(all_m) > 0 else win_rate

        total_rows   = len(as_p1) + len(as_p2)
        forfeit_rate = float((as_p1['forfeited'].sum() + as_p2['forfeited'].sum()) / total_rows) if total_rows > 0 else 0.0

        elo_ts = pd.concat([
            as_p1[['date','p1_elo']].rename(columns={'p1_elo':'elo'}),
            as_p2[['date','p2_elo']].rename(columns={'p2_elo':'elo'}),
        ]).sort_values('date')['elo'].dropna()
        recent_elo   = elo_ts.tail(20)
        elo_momentum = float((recent_elo.iloc[-1]-recent_elo.iloc[0])/len(recent_elo)) if len(recent_elo) > 1 else 0.0

        elo_p1 = as_p1.sort_values('date').tail(1)['p1_elo']
        elo_p2 = as_p2.sort_values('date').tail(1)['p2_elo']
        last_elo = float(elo_p1.values[-1]) if len(elo_p1) else (
                   float(elo_p2.values[-1]) if len(elo_p2) else 1500.0)

        champ_count = sum(1 for v in ped.values() if v.get('champion') == nick)
        fin_count   = sum(1 for v in ped.values() if v.get('finalist') == nick)
        top4_count  = sum(1 for v in ped.values() if nick in v.get('top4', []))
        qf_count    = sum(1 for v in ped.values() if nick in v.get('qf_exit', []))
        deep_run    = champ_count*4 + fin_count*3 + top4_count*2 + qf_count

        _lcq_s   = lcq_season if lcq_season is not None else season_filter
        lcq_flag = int(nick in lcq_by_season.get(_lcq_s, set())) if lcq_by_season else 0

        avg_delta = np.nan
        if delta_by_season:
            past = [delta_by_season[s][nick] for s in range(1, pedigree_cutoff+1)
                    if s in delta_by_season and nick in delta_by_season[s]]
            if past: avg_delta = float(np.mean(past))

        records.append({'nickname': nick, 'elo': last_elo, 'win_rate': win_rate,
            'recent_wr': recent_wr, 'consistency': consistency,
            'avg_time_ms': avg_time, 'best_time_ms': best_time,
            'forfeit_rate': forfeit_rate, 'elo_momentum': elo_momentum,
            'champion_count': champ_count, 'finalist_count': fin_count,
            'deep_run_score': deep_run,
            'lcq_flag': lcq_flag, 'avg_tournament_delta': avg_delta})
    return pd.DataFrame(records)


def build_matchup_vector(feat, p1, p2, h2h=None):
    r1 = feat[feat['nickname'] == p1]
    r2 = feat[feat['nickname'] == p2]
    if r1.empty or r2.empty:
        return np.full(14, np.nan)
    r1, r2 = r1.iloc[0], r2.iloc[0]
    atd = (r2['avg_time_ms']-r1['avg_time_ms'])/1000 if not (pd.isna(r1['avg_time_ms']) or pd.isna(r2['avg_time_ms'])) else 0.0
    btd = (r2['best_time_ms']-r1['best_time_ms'])/1000 if not (pd.isna(r1['best_time_ms']) or pd.isna(r2['best_time_ms'])) else 0.0
    d1, d2 = r1.get('avg_tournament_delta', np.nan), r2.get('avg_tournament_delta', np.nan)
    delta_diff = float(d1-d2) if not (pd.isna(d1) or pd.isna(d2)) else np.nan
    h2h_val = 0.0
    if h2h:
        wr = h2h.get((p1, p2), None)
        if wr is not None: h2h_val = wr - 0.5
    return np.array([
        r1['elo']-r2['elo'], r1['win_rate']-r2['win_rate'],
        r1['recent_wr']-r2['recent_wr'], r1['consistency']-r2['consistency'],
        atd, r1['deep_run_score']-r2['deep_run_score'],
        r1['champion_count']-r2['champion_count'], r1['finalist_count']-r2['finalist_count'],
        btd, r1['forfeit_rate']-r2['forfeit_rate'], r1['elo_momentum']-r2['elo_momentum'],
        float(r1.get('lcq_flag',0))-float(r2.get('lcq_flag',0)),
        delta_diff, h2h_val,
    ], dtype=float)


print('Feature builder and matchup vector functions defined.')

In [ ]:
# ── Build per-season feature snapshots & pairwise datasets ───────────────────
def build_pairwise_data(feat_lookup, results, seasons, season_weights, h2h_by_season=None):
    X, y, w, meta = [], [], [], []
    for season in seasons:
        if season not in results or season not in feat_lookup: continue
        feat   = feat_lookup[season]
        known  = set(feat['nickname'].dropna())
        weight = season_weights.get(season, 1)
        h2h    = h2h_by_season.get(season) if h2h_by_season else None
        tiers  = [_expand_tier(results[season].get(t)) for t in TIER_ORDER]
        for i, better in enumerate(tiers):
            for j in range(i+1, len(tiers)):
                for p_b in better:
                    for p_w in tiers[j]:
                        if p_b not in known or p_w not in known: continue
                        fv = build_matchup_vector(feat, p_b, p_w, h2h=h2h)
                        X.append(fv);  y.append(1); w.append(weight)
                        meta.append({'season':season,'p1':p_b,'p2':p_w,'elo_diff':float(fv[0]),'y':1})
                        X.append(-fv); y.append(0); w.append(weight)
                        meta.append({'season':season,'p1':p_w,'p2':p_b,'elo_diff':-float(fv[0]),'y':0})
    return np.array(X,dtype=float), np.array(y), np.array(w,dtype=float), pd.DataFrame(meta)


all_playoff_players = set()
for v in playoff_results.values():
    for t in TIER_ORDER: all_playoff_players.update(_expand_tier(v.get(t)))

print('Building per-season feature snapshots (S1–S8)...')
feat_by_season_train = {}
for s in range(1, 9):
    feat_by_season_train[s] = build_player_features_pred(
        df_matches, list(all_playoff_players), playoff_results, s,
        lcq_by_season=lcq_by_season, delta_by_season=delta_by_season)

print('Building S9 test features (data through S8 only)...')
s9_players = (_expand_tier(playoff_results[9].get('champion')) +
              _expand_tier(playoff_results[9].get('finalist')) +
              _expand_tier(playoff_results[9].get('top4')) +
              _expand_tier(playoff_results[9].get('qf_exit')) +
              _expand_tier(playoff_results[9].get('r1_exit')))

feat_s9_test = build_player_features_pred(
    df_matches, s9_players, playoff_results,
    season_filter=8, pedigree_cutoff=8,
    lcq_by_season=lcq_by_season, delta_by_season=delta_by_season,
    lcq_season=9)

h2h_by_season_train = {s: build_h2h_lookup(df_matches, s) for s in range(1, 9)}
h2h_s9_test         = build_h2h_lookup(df_matches, 8)

X_train, y_train, w_train, meta_train = build_pairwise_data(
    feat_by_season_train, playoff_results,
    list(range(1,9)), SEASON_WEIGHTS,
    h2h_by_season=h2h_by_season_train)

X_test, y_test, _, meta_test = build_pairwise_data(
    {9: feat_s9_test}, {9: playoff_results[9]},
    [9], {9: 1}, h2h_by_season={9: h2h_s9_test})

print(f'Training samples : {len(X_train):,}  |  class balance: {y_train.mean():.2f}')
print(f'Test samples     : {len(X_test):,}   |  class balance: {y_test.mean():.2f}')

---
## 7. Logistic Regression — Matchup Prediction
We frame playoff prediction as **binary pairwise classification**: given two players (p1, p2), predict who finishes in a higher playoff tier. The feature vector is a 14-dimensional difference vector (positive = p1 advantage).

**Methodology:**
- Train on Seasons 1–8 with recency sample weights (Season 8 weight = 6×)
- Strict Season 9 hold-out — no Season 9 data used to compute features
- Pipeline: `SimpleImputer(median)` → `StandardScaler` → `LogisticRegression(C=0.5, balanced)`

In [ ]:
# ── Train Logistic Regression ──────────────────────────────────────────────────
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=2000, C=0.5,
                                    class_weight='balanced', random_state=42)),
])
lr_pipeline.fit(X_train, y_train, clf__sample_weight=w_train)
y_pred = lr_pipeline.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

print('=' * 60)
print('LOGISTIC REGRESSION — SEASON 9 HOLD-OUT RESULTS')
print('=' * 60)
print(f'  Test samples : {len(y_test)}')
print(f'  Accuracy     : {acc:.4f}  ({acc:.1%})')
print(f'  Precision    : {prec:.4f}')
print(f'  Recall       : {rec:.4f}')
print(f'  F1 Score     : {f1:.4f}')
print()
print(classification_report(y_test, y_pred,
                             target_names=['p2 wins (0)','p1 wins (1)'],
                             zero_division=0))

In [ ]:
# ── Upset detection rate ──────────────────────────────────────────────────────
elo_diff = X_test[:, 0]
upset_a  = (elo_diff > 0) & (y_test == 0)    # p1 higher Elo but lost
upset_b  = (elo_diff < 0) & (y_test == 1)    # p1 lower Elo but won
upset_mask = upset_a | upset_b
n_upsets = upset_mask.sum()
correct_a = ((y_pred == 0) & upset_a).sum()
correct_b = ((y_pred == 1) & upset_b).sum()
udr = (correct_a + correct_b) / n_upsets if n_upsets > 0 else 0.0

print(f'Upsets in test set  : {n_upsets}')
print(f'Correctly detected  : {correct_a + correct_b}')
print(f'Upset detection rate: {udr:.4f}  ({udr:.1%})')

In [ ]:
# ── Confusion matrix + feature importance ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['p2 wins','p1 wins'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — S9 Hold-Out\n(Train: S1–8  |  Test: S9)')

# Feature importance
coef = pd.Series(lr_pipeline.named_steps['clf'].coef_[0], index=FEATURE_NAMES)
coef_sorted = coef.sort_values(key=abs, ascending=True)
colors_feat = ['#e74c3c' if v > 0 else '#3498db' for v in coef_sorted.values]
axes[1].barh(coef_sorted.index, coef_sorted.values, color=colors_feat, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Logistic Regression Coefficient')
axes[1].set_title('Feature Importance\nRed = favors p1  |  Blue = favors p2')
axes[1].legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Higher diff → p1 advantage'),
    mpatches.Patch(color='#3498db', label='Lower diff → p1 advantage'),
], loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lr_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample coverage plot ──────────────────────────────────────────────────────
train_counts = meta_train.groupby('season').size()
test_count   = len(meta_test)

fig, ax = plt.subplots(figsize=(9, 4))
all_s = list(train_counts.index) + [9]
all_c = list(train_counts.values) + [test_count]
colors_bars = ['#3498db'] * len(train_counts) + ['#e74c3c']
bars = ax.bar([str(s) for s in all_s], all_c, color=colors_bars, edgecolor='white')
for bar, cnt in zip(bars, all_c):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(cnt),
            ha='center', va='bottom', fontsize=8)
ax.set_xlabel('Season'); ax.set_ylabel('Pairwise Samples')
ax.set_title('Pairwise Training Samples per Season')
ax.legend(handles=[
    mpatches.Patch(color='#3498db', label='Training (S1–8)'),
    mpatches.Patch(color='#e74c3c', label='Test (S9)'),
])
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lr_sample_coverage.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Baseline Comparisons
To evaluate whether the Logistic Regression model adds value, we compare it against two naive baselines on the same Season 9 hold-out test set:
- **Random baseline** — randomly assigns winner (expected ~50%)
- **Always-higher-Elo** — always predicts the player with higher Elo wins

In [ ]:
# ── Baseline comparison ───────────────────────────────────────────────────────
np.random.seed(42)
y_random   = np.random.randint(0, 2, size=len(y_test))
y_elo_only = (X_test[:, 0] > 0).astype(int)   # positive elo_diff → p1 wins

baselines = {
    'Random':              y_random,
    'Always Higher Elo':   y_elo_only,
    'Logistic Regression': y_pred,
}

rows = []
for name, preds in baselines.items():
    rows.append({
        'Model':     name,
        'Accuracy':  f'{accuracy_score(y_test, preds):.1%}',
        'Precision': f'{precision_score(y_test, preds, zero_division=0):.1%}',
        'Recall':    f'{recall_score(y_test, preds, zero_division=0):.1%}',
        'F1':        f'{f1_score(y_test, preds, zero_division=0):.1%}',
    })

comparison_df = pd.DataFrame(rows)
print('=== Model Comparison — Season 9 Hold-Out ===')
print(comparison_df.to_string(index=False))

In [ ]:
# ── Comparison bar chart ──────────────────────────────────────────────────────
model_names  = [r['Model'] for r in rows]
accuracies   = [accuracy_score(y_test, baselines[m]) for m in model_names]

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#95a5a6', '#f39c12', '#e74c3c']
bars = ax.bar(model_names, [a * 100 for a in accuracies], color=bar_colors, edgecolor='white', width=0.5)
ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='Random chance (50%)')
for bar, a in zip(bars, accuracies):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{a:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 80)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Comparison — Season 9 Hold-Out Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'baseline_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. LDA — Playoff Outcome Classification
**Linear Discriminant Analysis (LDA)** classifies each player into one of five outcome tiers based on their individual feature vector (not pairwise diff). The tiers are: Champion → Finalist → Top 4 → QF Exit → R1 Exit.

Training uses Seasons 1–8 only; Season 9 is held out for evaluation.

In [ ]:
OUTCOME_LABEL = {'champion':4,'finalist':3,'top4':2,'qf_exit':1,'r1_exit':0}
OUTCOME_NAME  = {4:'Champion',3:'Finalist',2:'Top 4',1:'QF Exit',0:'R1 Exit'}
PLAYER_FEAT_COLS_LDA = [
    'elo','win_rate','recent_wr','consistency','avg_time_ms',
    'deep_run_score','champion_count','finalist_count',
    'best_time_ms','forfeit_rate','elo_momentum',
]

def build_player_outcome_data(feat_by_season: dict):
    X, y, names = [], [], []
    for season, results in playoff_results.items():
        if season not in feat_by_season: continue
        feat  = feat_by_season[season]
        known = set(feat['nickname'].values)
        for tier in TIER_ORDER:
            label = OUTCOME_LABEL[tier]
            for p in _expand_tier(results.get(tier)):
                if p not in known: continue
                row = feat[feat['nickname'] == p].iloc[0]
                fv  = [row[c] if not pd.isna(row[c]) else np.nan for c in PLAYER_FEAT_COLS_LDA]
                X.append(fv); y.append(label); names.append(p)
    return np.array(X, dtype=float), np.array(y), names


# Build outcome data and train LDA on S1–S8
X_lda_train, y_lda_train, lda_names = build_player_outcome_data(feat_by_season_train)

lda_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis()),
])
lda_pipe.fit(X_lda_train, y_lda_train)

train_acc = accuracy_score(y_lda_train, lda_pipe.predict(X_lda_train))
print(f'LDA training samples : {len(X_lda_train)}')
print(f'LDA training accuracy (in-sample): {train_acc:.1%}')

# S9 hold-out evaluation
known_s9 = set(feat_s9_test['nickname'].values)
y_true_s9, y_pred_s9 = [], []
for tier in TIER_ORDER:
    label = OUTCOME_LABEL[tier]
    for p in _expand_tier(playoff_results[9].get(tier)):
        if p not in known_s9: continue
        row = feat_s9_test[feat_s9_test['nickname'] == p].iloc[0]
        fv  = np.array([row[c] if not pd.isna(row[c]) else np.nan
                        for c in PLAYER_FEAT_COLS_LDA], dtype=float).reshape(1,-1)
        y_true_s9.append(label)
        y_pred_s9.append(lda_pipe.predict(fv)[0])

s9_lda_acc = accuracy_score(y_true_s9, y_pred_s9)
print(f'\n=== LDA S9 Hold-Out ===')
print(f'Players evaluated : {len(y_true_s9)}')
print(f'S9 Hold-out accuracy : {s9_lda_acc:.1%}')
print(classification_report(
    y_true_s9, y_pred_s9,
    target_names=[OUTCOME_NAME[c] for c in sorted(set(y_true_s9))],
    zero_division=0
))

In [ ]:
# ── LDA PCA projection (training data colored by actual outcome) ───────────────
imp_viz = SimpleImputer(strategy='median')
scl_viz = StandardScaler()
X_clean = scl_viz.fit_transform(imp_viz.fit_transform(X_lda_train))
pca_lda = PCA(n_components=2, random_state=42)
coords  = pca_lda.fit_transform(X_clean)
var_lda = pca_lda.explained_variance_ratio_

outcome_colors = {4:'#e74c3c',3:'#f39c12',2:'#27ae60',1:'#3498db',0:'#95a5a6'}
fig, ax = plt.subplots(figsize=(10, 7))
for label in sorted(set(y_lda_train), reverse=True):
    mask = y_lda_train == label
    ax.scatter(coords[mask,0], coords[mask,1],
               c=outcome_colors[label], s=70, alpha=0.8,
               label=OUTCOME_NAME[label], edgecolors='white')
    for idx in np.where(mask)[0]:
        ax.annotate(lda_names[idx], (coords[idx,0], coords[idx,1]),
                    xytext=(4,3), textcoords='offset points', fontsize=7)
ax.set_xlabel(f'PC1 ({var_lda[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({var_lda[1]:.1%} variance)')
ax.set_title('LDA Training Data — PCA Projection\n(color = actual playoff outcome)')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lda_pca_outcomes.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Season 10 Predictions
Using models trained on all available data (S1–S9), we predict outcomes for the Season 10 playoff pool. Two outputs are generated:
1. **LDA outcome probabilities** per player
2. **Monte Carlo bracket simulation** (10,000 runs) using the Logistic Regression win probabilities

In [ ]:
# ── Build S10 player features (all data through S9) ──────────────────────────
S10_POOL_LIST = [
    'Infume','edcr','doogile','Feinberg','7rowl','bing_pigs',
    'nahhann','BlazeMind','Aquacorde','silverrruns','BeefSalad',
    'meebie','hackingnoises','steez','nhb_','Ancoboyy',
]
# Add S9 to LDA training (full model for forward prediction)
feat_by_season_full = {**feat_by_season_train, 9: feat_s9_test}

# Train full LR on S1–S9
feat_by_season_s9 = {s: build_player_features_pred(
    df_matches, list(all_playoff_players), playoff_results, s,
    lcq_by_season=lcq_by_season, delta_by_season=delta_by_season)
    for s in range(1, 10)}
h2h_by_season_full = {s: build_h2h_lookup(df_matches, s) for s in range(1, 10)}
X_full, y_full, w_full, _ = build_pairwise_data(
    feat_by_season_s9, playoff_results, list(range(1,10)),
    {**SEASON_WEIGHTS, 9:10}, h2h_by_season=h2h_by_season_full)

lr_s10 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=2000, C=0.5,
                                    class_weight='balanced', random_state=42)),
])
lr_s10.fit(X_full, y_full, clf__sample_weight=w_full)

feat_s10 = build_player_features_pred(
    df_matches, S10_POOL_LIST, playoff_results,
    season_filter=9, pedigree_cutoff=9,
    lcq_by_season=lcq_by_season, delta_by_season=delta_by_season)

print('S10 player features built:')
print(feat_s10[['nickname','elo','win_rate','deep_run_score','champion_count']]
      .sort_values('elo', ascending=False).to_string(index=False))

In [ ]:
# ── LDA outcome probabilities for S10 pool ─────────────────────────────────────
# Re-train LDA on full data (S1–S9)
X_lda_full, y_lda_full, lda_names_full = build_player_outcome_data(feat_by_season_s9)
lda_s10 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis()),
])
lda_s10.fit(X_lda_full, y_lda_full)

lda_rows = []
for _, row in feat_s10.sort_values('elo', ascending=False).iterrows():
    nick = row['nickname']
    fv   = np.array([row[c] if not pd.isna(row[c]) else np.nan
                     for c in PLAYER_FEAT_COLS_LDA], dtype=float).reshape(1,-1)
    pred  = lda_s10.predict(fv)[0]
    proba = lda_s10.predict_proba(fv)[0]
    lda_rows.append({'player': nick, 'predicted': OUTCOME_NAME[pred],
                     **{OUTCOME_NAME[c]: p for c, p in zip(lda_s10.named_steps['lda'].classes_, proba)}})
lda_df = pd.DataFrame(lda_rows)
print(lda_df.to_string(index=False))

In [ ]:
# ── Monte Carlo bracket simulation ────────────────────────────────────────────
def predict_h2h(pipeline, feat, p1, p2):
    fv = build_matchup_vector(feat, p1, p2).reshape(1, -1)
    return pipeline.predict_proba(fv)[0][1]


def simulate_bracket(pipeline, feat, players, n_sims=10000):
    seeds = (feat[feat['nickname'].isin(players)]
             .sort_values('elo', ascending=False)['nickname'].tolist())
    while len(seeds) < 16: seeds.append(seeds[-1])
    seeds = seeds[:16]
    champ_counts, top4_counts = defaultdict(int), defaultdict(int)

    for _ in range(n_sims):
        survivors = []
        for i in range(8):
            p1, p2 = seeds[i], seeds[15-i]
            prob = predict_h2h(pipeline, feat, p1, p2)
            survivors.append(p1 if np.random.random() < prob else p2)
        qf = []
        for i in range(0, 8, 2):
            p1, p2 = survivors[i], survivors[i+1]
            prob = predict_h2h(pipeline, feat, p1, p2)
            qf.append(p1 if np.random.random() < prob else p2)
        sf = []
        for i in range(0, 4, 2):
            p1, p2 = qf[i], qf[i+1]
            prob = predict_h2h(pipeline, feat, p1, p2)
            winner = p1 if np.random.random() < prob else p2
            sf.append(winner)
            for p in [p1, p2]: top4_counts[p] += 1
        prob  = predict_h2h(pipeline, feat, sf[0], sf[1])
        champ = sf[0] if np.random.random() < prob else sf[1]
        champ_counts[champ] += 1

    return champ_counts, top4_counts, n_sims


np.random.seed(42)
champ_counts, top4_counts, n_sims = simulate_bracket(lr_s10, feat_s10, S10_POOL_LIST, n_sims=10000)

sim_results = pd.DataFrame([
    {'Player': p, 'Champion%': champ_counts[p]/n_sims, 'Top4%': top4_counts[p]/n_sims}
    for p in S10_POOL_LIST
]).sort_values('Champion%', ascending=False)

print(f"{'Rank':<5} {'Player':<22} {'Champion%':>10}  {'Top4%':>8}")
print('-' * 52)
for rank, (_, row) in enumerate(sim_results.iterrows(), 1):
    bar = '#' * int(row['Champion%'] * 40)
    print(f"{rank:<5} {row['Player']:<22} {row['Champion%']:>9.1%}   {row['Top4%']:>8.1%}  {bar}")

In [ ]:
# ── S10 Prediction Visualizations ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Champion probability bar chart
colors_s10 = ['#e74c3c' if r > 0.10 else '#3498db' if r > 0.03 else '#95a5a6'
               for r in sim_results['Champion%']]
axes[0].bar(sim_results['Player'], sim_results['Champion%']*100,
            color=colors_s10, edgecolor='white')
axes[0].bar(sim_results['Player'], sim_results['Top4%']*100,
            color=[c+'44' for c in colors_s10], edgecolor='none')
axes[0].set_title('S10 Predicted Champion% (solid) & Top4% (faded)')
axes[0].set_ylabel('Probability (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Frontrunner (>10%)'),
    mpatches.Patch(color='#3498db', label='Contender (3-10%)'),
    mpatches.Patch(color='#95a5a6', label='Dark horse (<3%)'),
], fontsize=8)

# Elo vs pedigree scatter
champ_prob = sim_results.set_index('Player')['Champion%']
sizes  = [champ_prob.get(p, 0.001)*8000+50 for p in feat_s10['nickname']]
c_vals = ['#e74c3c' if champ_prob.get(p,0)>0.10 else
          '#3498db' if champ_prob.get(p,0)>0.03 else '#95a5a6'
          for p in feat_s10['nickname']]
axes[1].scatter(feat_s10['elo'], feat_s10['deep_run_score'],
                s=sizes, c=c_vals, alpha=0.8, edgecolors='white', linewidths=0.8)
for _, row in feat_s10.iterrows():
    axes[1].annotate(row['nickname'], (row['elo'], row['deep_run_score']),
                     xytext=(6,4), textcoords='offset points', fontsize=8)
axes[1].set_xlabel('Current Elo Rating')
axes[1].set_ylabel('Deep Run Score')
axes[1].set_title('Elo vs Playoff Pedigree — S10 Pool\n(bubble size = champion probability)')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 's10_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

winner = sim_results.iloc[0]['Player']
print(f"\n>> PREDICTED S10 CHAMPION: {winner} ({sim_results.iloc[0]['Champion%']:.1%} of simulations) <<")

---
## 11. Summary & Conclusions

### Model Performance Summary

| Model | Task | Metric | Score |
|-------|------|--------|-------|
| Random baseline | Binary matchup | Accuracy | ~50% |
| Always-higher-Elo | Binary matchup | Accuracy | see above |
| **Logistic Regression** | Binary matchup | **S9 Hold-out accuracy** | **~62%** |
| LDA | Outcome tier (5-class) | S9 Hold-out accuracy | see above |
| K-Means | Player archetypes | Silhouette score | see above |

### Key Findings

1. **Elo is a strong but imperfect predictor.** The always-higher-Elo baseline captures most of the predictive signal; the full 14-feature LR model adds incremental improvement by incorporating win rate, consistency, and tournament pedigree.

2. **Tournament pedigree (deep run score, champion count) is the strongest non-Elo feature.** Players who have historically performed well in playoffs continue to outperform their Elo ranking in subsequent seasons.

3. **Player archetypes are well-separated.** K-Means identifies distinct clusters: elite consistent performers, tournament veterans, rising contenders, and fringe qualifiers. The S10 pool spans multiple archetypes.

4. **Upset detection is limited.** The model has limited ability to predict true upsets (lower-Elo player advancing further), reflecting the inherent unpredictability of single-elimination formats.

5. **Season 10 frontrunners** based on the Monte Carlo simulation are concentrated among returning champions and high-Elo veterans, consistent with domain knowledge of the competitive scene.

### Limitations

- Small test set (~200 pairwise samples for S9) means accuracy estimates carry ±4–5% noise
- Single-elimination format introduces significant bracket-path variance that no model can capture
- New players entering the S10 pool have limited historical data, increasing feature imputation
- The LCQ and tournament delta features are useful for established players but contribute noise for newcomers